# Car Rental Validation Cases (Selected)
This notebook contains extracted validation sections from the cleaning notebook: 7, 8, 10, 11, 12, 13, 14, 15, 19, and 20.


In [1]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv("../Datasets/car_rental_dirty_dataset_new1.csv")

for col in ["Booking_TS", "Pickup_TS", "Return_TS"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

for col in ["Odo_Start", "Odo_End", "Rate"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")

if "Fuel_Level" in df.columns:
    fuel = df["Fuel_Level"].astype(str).str.replace("%", "", regex=False)
    df["Fuel_Level"] = pd.to_numeric(fuel, errors="coerce")
    df.loc[df["Fuel_Level"] > 1, "Fuel_Level"] = df.loc[df["Fuel_Level"] > 1, "Fuel_Level"] / 100


C:\Users\KIIT\AppData\Local\Temp\ipykernel_5356\3887982489.py:9: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")


## 7. Duplicate reservation dedup (same ID).


In [2]:
print(f"Duplicate Reservation_IDs: {df['Reservation_ID'].duplicated().sum()}")
df = df.drop_duplicates(subset='Reservation_ID', keep='first')
print(f"Shape after dedup: {df.shape}")

Duplicate Reservation_IDs: 473
Shape after dedup: (4527, 21)


## 8. Return before pickup rule validation.


In [3]:
swap_mask = df["Return_TS"] < df["Pickup_TS"]

df.loc[swap_mask, ["Pickup_TS","Return_TS"]] = \
df.loc[swap_mask, ["Return_TS","Pickup_TS"]].values

In [4]:
df["Duration_Hours"] = (df["Return_TS"] - df["Pickup_TS"]).dt.total_seconds() / 3600

In [5]:
invalid_duration = df[df["Duration_Hours"] < 0]

In [6]:
equal_mask = df["Return_TS"] == df["Pickup_TS"]

In [7]:
df.loc[equal_mask, ["Pickup_TS", "Return_TS"]] = pd.NaT

In [8]:
print(invalid_duration)

Empty DataFrame
Columns: [Reservation_ID, Customer_ID, Vehicle_ID, Vehicle_Class, Booking_Status, Booking_TS, Pickup_TS, Return_TS, Odo_Start, Odo_End, Fuel_Level, Rate, Promo_Code, City, GPS_Lat, GPS_Lon, Speed, Payment, Driver_License, Damage_Flag, Notes, Duration_Hours]
Index: []

[0 rows x 22 columns]


In [9]:
df.head(20)

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,...,Promo_Code,City,GPS_Lat,GPS_Lon,Speed,Payment,Driver_License,Damage_Flag,Notes,Duration_Hours
0,RES-00001,CUST-1675,CAR-014,Hatchback,No_Show,2025-02-13 07:56:00,2025-02-18 07:56:00,2025-02-18 09:56:00,57815,58065,...,NaN,New Delhi,12.908959,77.604003,133kmh,cash,DL-2699413330,Major,Customer satisfied with ride,2.0
1,RES-00002,CUST-0853,CAR230,SUV,Cancelled,NaT,NaT,NaT,77762,77766,...,NEW10,mumbai,12.946286,77.586710,55,CASH,DL-9806685378,Major,Customer requested early pickup,NaN
2,RES-00003,CUST-1181,CAR-001,Suzuki,Completed,NaT,NaT,2025-03-12 23:07:00,57856,58238,...,DISC20,delhi,12.914102,77.627323,128 km/h,upi,DL-9542219877,Minor,Customer reported minor scratch on door,NaN
3,RES-00004,CUST-0182,CAR-434,Suzuki,Cancelled,NaT,NaT,2025-01-11 22:16:00,59967,60239,...,SAVE50,delhi,12.873192,77.635268,25 km/h,cash,DL-4380657786,NaN,Customer requested early pickup,NaN
4,RES-00005,CUST-0356,CAR-121,Suzuki,No_Show,NaT,2025-05-09 04:22:00,NaT,24261,24524,...,DISC20,bangalore,12.919640,77.598891,26kmh,-,DL-6429812852,Minor,AC performance slightly low,NaN
5,RES-00006,CUST-0061,CAR_360,Luxury,Cancelled,NaT,2025-05-30 08:10:00,2025-05-31 09:10:00,36184,36211,...,DISC20,chennai,12.872343,77.588487,100 km/h,card,DL-7859136756,Major,Customer reported minor scratch on door,25.0
6,RES-00007,CUST-0793,CAR-114,Creta,Completed,NaT,NaT,NaT,41914,41881,...,WELCOME5,NaN,12.947693,77.603382,49,cash,DL-3024014652,Major,Tyre pressure alert during trip,NaN
7,RES-00008,CUST-0022,CAR_392,Luxury,No_Show,NaT,NaT,NaT,34930,35217,...,NEW10,Mumbai,12.927285,77.615525,fast,wallet,DL-2212993750,Major,AC performance slightly low,NaN
8,RES-00009,CUST-1700,car-505,Suzuki,No_Show,NaT,2025-04-17 12:30:00,NaT,66433,66519,...,SAVE50,Bengaluru,12.922702,77.643999,135,netbanking,DL-1593968320,Minor,Low fuel warning observed,NaN
9,RES-00010,CUST-1586,CAR-448,EV,Cancelled,NaT,NaT,NaT,66231,66471,...,DISC20,New Delhi,12.885247,77.596902,fast,netbanking,DL-9281443122,Major,Customer satisfied with ride,NaN


## 10. Mileage sanity check (End ≥ Start)


In [10]:
mileage_check = df[['Reservation_ID','Odo_Start','Odo_End']].copy()

# Swap Odo_Start and Odo_End where End < Start (likely data entry error)
odo_swap_mask = df['Odo_End'] < df['Odo_Start']
df.loc[odo_swap_mask, ['Odo_Start', 'Odo_End']] = df.loc[odo_swap_mask, ['Odo_End', 'Odo_Start']].values
print(f"Swapped Odo_Start/Odo_End for {odo_swap_mask.sum()} rows")

# Verify no invalid mileages remain
mileage_check = df[['Reservation_ID','Odo_Start','Odo_End']].copy()
mileage_check['Mileage_Valid'] = mileage_check['Odo_End'] >= mileage_check['Odo_Start']
print(f"Invalid mileages remaining: {(~mileage_check['Mileage_Valid']).sum()}")
mileage_check.head(10)

Swapped Odo_Start/Odo_End for 745 rows
Invalid mileages remaining: 0


,Reservation_ID,Odo_Start,Odo_End,Mileage_Valid
0,RES-00001,57815,58065,True
1,RES-00002,77762,77766,True
2,RES-00003,57856,58238,True
3,RES-00004,59967,60239,True
4,RES-00005,24261,24524,True
5,RES-00006,36184,36211,True
6,RES-00007,41881,41914,True
7,RES-00008,34930,35217,True
8,RES-00009,66433,66519,True
9,RES-00010,66231,66471,True


## 11. Refueling event detection vs fuel change and distance.


In [11]:
# Step 1: Normalize Fuel_Level (re-normalize in case of any remaining % values)
df["Fuel_Level"] = df["Fuel_Level"].astype(str).str.replace("%", "", regex=False)
df["Fuel_Level"] = pd.to_numeric(df["Fuel_Level"], errors="coerce")

# Convert percentage values to fraction
df.loc[df["Fuel_Level"] > 1, "Fuel_Level"] = df["Fuel_Level"] / 100

# Step 2: Prepare validity mask and distance driven directly on main dataframe
valid_odo_mask = df["Odo_End"] >= df["Odo_Start"]
df["Distance_Driven"] = pd.NA
df.loc[valid_odo_mask, "Distance_Driven"] = (
    df.loc[valid_odo_mask, "Odo_End"] - df.loc[valid_odo_mask, "Odo_Start"]
)

# Step 3: Detect refuel events directly on main dataframe
sorted_idx = df.sort_values(["Vehicle_ID", "Odo_Start"]).index
fuel_diff = df.loc[sorted_idx].groupby("Vehicle_ID")["Fuel_Level"].diff()
refuel_flag = (fuel_diff > 0).map({True: "Refueled", False: "No Refuel"})

df["Refuel_Event"] = pd.NA
df.loc[refuel_flag.index, "Refuel_Event"] = refuel_flag
df.loc[~valid_odo_mask, "Refuel_Event"] = pd.NA

# Step 4: Display
df.head(10)
df['Refuel_Event'].unique()

array(['No Refuel', 'Refueled'], dtype=object)

## 12. Vehicle availability overlap checks.


In [12]:
df = df.sort_values(['Vehicle_ID','Pickup_TS'])

df['Prev_Return'] = df.groupby('Vehicle_ID')['Return_TS'].shift(1)

df['Overlap_Flag'] = df['Pickup_TS'] < df['Prev_Return']

df['Overlap_Flag'].value_counts()

Overlap_Flag
False    4524
True        3
Name: count, dtype: int64

## 13. Damage / Incident Log Linkage to Reservation


In [13]:
#inspect
df[['Reservation_ID','Vehicle_ID','Customer_ID','Damage_Flag','Notes']].head()

#Extract rows where damage was reported
damage_logs = df[
    df['Damage_Flag'].notna() &
    (df['Damage_Flag'] != 'None')
]


#Create a separate incident log dataset
incident_log = damage_logs[[
    'Reservation_ID',
    'Vehicle_ID',
    'Customer_ID',
    'Damage_Flag',
    'Notes'
]].copy()


#Check for records where damage is flagged but notes are missing
missing_notes = incident_log[
    incident_log['Notes'].isna() |
    (incident_log['Notes'].str.strip() == "")
]


print("Damage incidents missing description:", len(missing_notes))


#Detect possible damage mentioned in notes but flag is 'None'
possible_damage = df[
    (df['Damage_Flag'] == 'None') &
    (df['Notes'].str.contains('damage|scratch|dent|crack', case=False, na=False))
]


print("Possible misclassified incidents:", len(possible_damage))


#Display the final incident log dataset
incident_log.head(15)
df['Notes'].unique()


Damage incidents missing description: 500
Possible misclassified incidents: 0


<StringArray>
[                                      nan,
               'Low fuel warning observed',
            'Customer satisfied with ride',
    'Vehicle returned late due to traffic',
          'Car returned in good condition',
             'AC performance slightly low',
         'Tyre pressure alert during trip',
         'Customer requested early pickup',
 'Customer reported minor scratch on door',
              'Interior cleaning required',
  'Navigation system malfunction reported']
Length: 11, dtype: str

## 14. Driver license masking and validation (if present).


In [14]:
#validation
def validate_license(x):
    if pd.isna(x):
        return None
    
    x = str(x)
    
    if re.match(r"DL-\d{10}$", x):
        return x
    else:
        return None

df["Driver_License"] = df["Driver_License"].apply(validate_license)


In [15]:
#masking
def mask_license(x):
    if pd.isna(x):
        return None
    
    return x[:7] + "XXXXXX"

df["Driver_License"] = df["Driver_License"].apply(mask_license)

## 15. Promo/coupon code validation & expiry checks.


In [16]:
# convert pickup timestamp
df["Pickup_TS"] = pd.to_datetime(df["Pickup_TS"], errors="coerce")

# promo expiry dictionary
promo_expiry = {
    "NEW10": "2026-03-31",
    "DISC20": "2026-02-28",
    "SAVE50": "2026-06-30",
    "WELCOME5": "2026-01-31"
}

# convert expiry values to datetime
promo_expiry = {k: pd.to_datetime(v) for k, v in promo_expiry.items()}

# map expiry date to dataset
df["Promo_Expiry"] = df["Promo_Code"].map(promo_expiry)

df["Promo_Status"] = "Valid"

# invalid promo codes
df.loc[df["Promo_Code"].notna() & df["Promo_Expiry"].isna(), "Promo_Status"] = "Invalid_Code"

# expired promo codes
df.loc[(df["Promo_Expiry"].notna()) & (df["Pickup_TS"] > df["Promo_Expiry"]), "Promo_Status"] = "Expired"



In [17]:
df.head(30)


,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_Status,Booking_TS,Pickup_TS,Return_TS,Odo_Start,Odo_End,...,Driver_License,Damage_Flag,Notes,Duration_Hours,Distance_Driven,Refuel_Event,Prev_Return,Overlap_Flag,Promo_Expiry,Promo_Status
3723,RES-03724,CUST-0023,CAR-010,Sedan,Cancelled,2025-04-05 02:12:00,2025-04-12 02:12:00,NaT,22602,22668,...,DL-7956XXXXXX,Major,NaN,NaN,66,No Refuel,NaT,False,2026-03-31,Valid
961,RES-00962,CUST-0614,CAR-010,Sedan,Completed,NaT,2025-06-02 20:29:00,2025-06-02 23:29:00,11555,11908,...,DL-8050XXXXXX,Major,NaN,3.0,353,No Refuel,NaT,False,2026-01-31,Valid
338,RES-00339,CUST-1270,CAR-010,Sedan,Completed,NaT,NaT,NaT,15111,15127,...,DL-7942XXXXXX,Major,Low fuel warning observed,NaN,16,No Refuel,2025-06-02 23:29:00,False,2026-02-28,Valid
1888,RES-01889,CUST-1361,CAR-010,Sedan,Cancelled,NaT,NaT,2025-04-24 06:09:00,59819,59936,...,DL-6181XXXXXX,Minor,Customer satisfied with ride,NaN,117,No Refuel,NaT,False,2026-06-30,Valid
2090,RES-02091,CUST-0310,CAR-010,Sedan,Completed,NaT,NaT,2025-03-04 13:28:00,51859,52188,...,DL-7627XXXXXX,NaN,Vehicle returned late due to traffic,NaN,329,Refueled,2025-04-24 06:09:00,False,2026-06-30,Valid
2374,RES-02375,CUST-0328,CAR-010,Sedan,Cancelled,2025-01-02 08:55:00,NaT,2025-01-06 18:55:00,32490,32620,...,DL-3101XXXXXX,NaN,Vehicle returned late due to traffic,NaN,130,No Refuel,2025-03-04 13:28:00,False,2026-01-31,Valid
3242,RES-03243,CUST-1164,CAR-010,Sedan,Cancelled,NaT,NaT,2025-06-05 12:01:00,46923,46958,...,DL-2242XXXXXX,Major,Car returned in good condition,NaN,35,Refueled,2025-01-06 18:55:00,False,2026-02-28,Valid
3775,RES-03776,CUST-0430,CAR-010,Sedan,No_Show,NaT,NaT,NaT,65131,65624,...,DL-2967XXXXXX,Minor,NaN,NaN,493,Refueled,2025-06-05 12:01:00,False,2026-03-31,Valid
4386,RES-04387,CUST-0081,CAR-010,Sedan,Completed,NaT,NaT,NaT,34051,34468,...,DL-3155XXXXXX,Minor,AC performance slightly low,NaN,417,No Refuel,NaT,False,2026-06-30,Valid
0,RES-00001,CUST-1675,CAR-014,Hatchback,No_Show,2025-02-13 07:56:00,2025-02-18 07:56:00,2025-02-18 09:56:00,57815,58065,...,DL-2699XXXXXX,Major,Customer satisfied with ride,2.0,250,No Refuel,NaT,False,NaT,Valid


## 19. Rate plan mapping to master tariffs.


In [18]:
df['Rate'] = df['Rate'].astype(str)
df['Rate'] = df['Rate'].str.replace('[^0-9.]', '', regex=True)
df['Rate'] = pd.to_numeric(df['Rate'], errors='coerce')

## 20. Tax/GST computation validation.


In [19]:
#USE CASE 20: Tax/GST computation validation.

df["Rate"] = pd.to_numeric(df["Rate"], errors="coerce")
df["Total_Amount"] = (df["Rate"] * 1.18).round(2)